In [0]:
%python
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_passenger') AS
WITH geo AS (
  SELECT n.n_nationkey AS nation_key, n.n_name AS nation, r.r_name AS region
  FROM IDENTIFIER(:catalog || '.silver.nation') n
  JOIN IDENTIFIER(:catalog || '.silver.region') r ON n.n_regionkey = r.r_regionkey
)
SELECT
  ROW_NUMBER() OVER (ORDER BY p.pa_passengerid) AS passenger_key,
  p.pa_passengerid AS passenger_id,
  p.pa_name AS passenger_name,
  p.pa_address AS address,
  p.pa_phone AS phone,
  p.pa_loyaltypoints AS loyalty_points,
  p.pa_travelpurpose AS travel_purpose,
  g.nation AS passenger_nation,
  g.region AS passenger_region
FROM IDENTIFIER(:catalog || '.silver.passenger') p
LEFT JOIN geo g ON p.pa_nationkey = g.nation_key

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_route') AS
SELECT
  ROW_NUMBER() OVER (ORDER BY ro_routeid) AS route_key,
  ro_routeid AS route_id,
  ro_name AS route_name,
  ro_aircraft AS aircraft,
  ro_airline_brand AS airline_brand,
  ro_routetype AS route_type,
  ro_capacity AS capacity,
  ro_cabinconfig AS cabin_config,
  ro_baseprice AS base_price
FROM IDENTIFIER(:catalog || '.silver.route')

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_airline') AS
WITH geo AS (
  SELECT n.n_nationkey AS nation_key, n.n_name AS nation, r.r_name AS region
  FROM IDENTIFIER(:catalog || '.silver.nation') n
  JOIN IDENTIFIER(:catalog || '.silver.region') r ON n.n_regionkey = r.r_regionkey
)
SELECT
  ROW_NUMBER() OVER (ORDER BY a.al_airlineid) AS airline_key,
  a.al_airlineid AS airline_id,
  a.al_name AS airline_name,
  a.al_address AS address,
  a.al_phone AS phone,
  a.al_rating AS rating,
  g.nation AS airline_nation,
  g.region AS airline_region
FROM IDENTIFIER(:catalog || '.silver.airline') a
LEFT JOIN geo g ON a.al_nationkey = g.nation_key

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.dim_geography') AS
SELECT
  n.n_nationkey AS nation_key,
  n.n_name AS nation_name,
  n.n_comment AS nation_comment,
  r.r_regionkey AS region_key,
  r.r_name AS region_name,
  r.r_comment AS region_comment
FROM IDENTIFIER(:catalog || '.silver.nation') n
JOIN IDENTIFIER(:catalog || '.silver.region') r ON n.n_regionkey = r.r_regionkey

In [0]:
-- grain: one row per flight-segment booking line; net_revenue = ticket_price * (1 - discount), before tax
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.fact_bookings') AS
WITH base AS (
  SELECT
    fb.fb_bookingid, fb.fb_linenumber, fb.fb_routeid, fb.fb_airlineid,
    fb.fb_seats, fb.fb_ticketprice, fb.fb_discount, fb.fb_tax,
    fb.fb_refundflag, fb.fb_segmentstatus, fb.fb_departuredate, fb.fb_arrivaldate, fb.fb_commitdate, fb.fb_cabinclass,
    b.bk_passengerid, b.bk_status, b.bk_bookingdate, b.bk_flightdate, b.bk_bookingpriority, b.bk_agent
  FROM IDENTIFIER(:catalog || '.silver.flight_booking') fb
  JOIN IDENTIFIER(:catalog || '.silver.booking') b ON fb.fb_bookingid = b.bk_bookingid
)
SELECT
  bs.fb_bookingid AS booking_id,
  bs.fb_linenumber AS line_number,
  dp.passenger_key,
  bs.bk_passengerid AS passenger_id,
  dr.route_key,
  bs.fb_routeid AS route_id,
  da.airline_key,
  bs.fb_airlineid AS airline_id,
  bs.bk_bookingdate AS booking_date,
  bs.bk_flightdate AS flight_date,
  bs.fb_departuredate AS departure_date,
  bs.fb_arrivaldate AS arrival_date,
  bs.fb_commitdate AS commit_date,
  YEAR(bs.bk_flightdate) AS flight_year,
  bs.bk_status AS booking_status,
  bs.bk_bookingpriority AS booking_priority,
  bs.bk_agent AS agent,
  bs.fb_cabinclass AS cabin_class,
  bs.fb_segmentstatus AS segment_status,
  bs.fb_refundflag AS refund_flag,
  bs.fb_seats AS seats,
  bs.fb_ticketprice AS ticket_price,
  bs.fb_discount AS discount,
  bs.fb_tax AS tax,
  ROUND(bs.fb_ticketprice * bs.fb_seats, 2) AS gross_amount,
  bs.fb_ticketprice * (1 - bs.fb_discount) AS net_revenue,
  dp.passenger_nation,
  dp.passenger_region,
  da.airline_nation,
  da.airline_region
FROM base bs
LEFT JOIN IDENTIFIER(:catalog || '.gold.dim_passenger') dp ON bs.bk_passengerid = dp.passenger_id
LEFT JOIN IDENTIFIER(:catalog || '.gold.dim_route') dr ON bs.fb_routeid = dr.route_id
LEFT JOIN IDENTIFIER(:catalog || '.gold.dim_airline') da ON bs.fb_airlineid = da.airline_id

In [0]:
CREATE OR REPLACE VIEW IDENTIFIER(:catalog || '.gold.vw_annual_booking_kpis') AS
SELECT
  flight_year,
  ROUND(SUM(net_revenue), 2) AS annual_net_revenue,
  COUNT(DISTINCT booking_id) AS distinct_bookings,
  COUNT(DISTINCT passenger_id) AS active_passengers,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT booking_id), 2) AS avg_booking_value
FROM IDENTIFIER(:catalog || '.gold.fact_bookings')
GROUP BY flight_year

In [0]:
-- independent Silver calc vs Gold totals
WITH gold_tot AS (
  SELECT COUNT(*) AS gold_rows, ROUND(SUM(net_revenue), 2) AS gold_revenue
  FROM IDENTIFIER(:catalog || '.gold.fact_bookings')
),
silver_tot AS (
  SELECT COUNT(*) AS silver_rows, ROUND(SUM(fb_ticketprice * (1 - fb_discount)), 2) AS silver_revenue
  FROM IDENTIFIER(:catalog || '.silver.flight_booking')
)
SELECT
  gold_rows, silver_rows, gold_rows - silver_rows AS row_diff,
  gold_revenue, silver_revenue, ROUND(gold_revenue - silver_revenue, 2) AS revenue_diff,
  CASE WHEN gold_rows = silver_rows AND gold_revenue = silver_revenue THEN 'PASS' ELSE 'FAIL' END AS status
FROM gold_tot, silver_tot

In [0]:
-- each dim: rows must equal distinct keys
SELECT 'dim_passenger' AS dimension, COUNT(*) AS total_rows, COUNT(DISTINCT passenger_key) AS distinct_keys,
  CASE WHEN COUNT(*) = COUNT(DISTINCT passenger_key) THEN 'PASS' ELSE 'FAIL' END AS status
FROM IDENTIFIER(:catalog || '.gold.dim_passenger')
UNION ALL
SELECT 'dim_route', COUNT(*), COUNT(DISTINCT route_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT route_key) THEN 'PASS' ELSE 'FAIL' END
FROM IDENTIFIER(:catalog || '.gold.dim_route')
UNION ALL
SELECT 'dim_airline', COUNT(*), COUNT(DISTINCT airline_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT airline_key) THEN 'PASS' ELSE 'FAIL' END
FROM IDENTIFIER(:catalog || '.gold.dim_airline')
UNION ALL
SELECT 'dim_geography', COUNT(*), COUNT(DISTINCT nation_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT nation_key) THEN 'PASS' ELSE 'FAIL' END
FROM IDENTIFIER(:catalog || '.gold.dim_geography')